# Session 5 - rolling-origin validation, hyperparameter tuning, sparse-segment investigation

**Third pass, after two rounds of external review, before anything was committed.** Round 1
added the rolling-origin MA4 comparison that was missing. Round 2 caught a subtler bug this
notebook now fixes: `eval_metric=vn1_lgb_eval` alone does NOT make early stopping exclusively
driven by the VN1 score -- LightGBM keeps its own built-in metric (e.g. "l2" for
`objective="regression"`) registered alongside it unless `metric="None"` is set, and without
`first_metric_only=True` on the callback, both metrics can influence which iteration is chosen.
Verified directly before fixing (see `src/models/lightgbm_model.py::train_lgbm`, which now
asserts the tracked metric list explicitly rather than assuming the fix worked).

Fixing this materially changed several conclusions from the first two passes -- not just the
headline number. See DECISIONS.md D009 for the full, corrected account.

Per DECISIONS.md D008: the true 13-week holdout is not used for any selection here, only for
one final evaluation at the end -- and even that is a development holdout (already inspected
in Sessions 3, 4, and both earlier passes of this session), not a pristine test set.

In [1]:
import os
from pathlib import Path

current = Path.cwd()
while not (current / "CONTEXT.md").is_file():
    if current == current.parent:
        raise FileNotFoundError("Could not find CONTEXT.md in any parent directory")
    current = current.parent
os.chdir(current)
print("repo root:", Path.cwd())

import numpy as np
import pandas as pd
import lightgbm as lgb
print("lightgbm version:", lgb.__version__)

from src.data import KEY
from src.features import build_direct_horizon_matrix, purge_fit_origins, rolling_origin_folds
from src.backtest import (
    split_last_horizon, ma12_forecast, zero_forecast, seasonal_naive_forecast,
    score_forecast, HORIZON,
)
from src.metrics import vn1_score, vn1_score_breakdown, vn1_score_by_group
from src.models.lightgbm_model import train_lgbm

df = pd.read_parquet("data/processed/vn1_long.parquet")
train, test = split_last_horizon(df, horizon=HORIZON)

repo root: d:\vn1-forecasting


lightgbm version: 4.7.0


## 1. Rolling-origin validation folds

`rolling_origin_folds()` (src/features.py): 3 walk-forward, purged folds (D008) inside the
same 52-140 origin pool Session 4 used.

In [2]:
# Cell - build the 3 rolling-origin folds and pre-build their feature matrices once
origins = list(range(52, 144, 4))
horizons = list(range(1, 14))
folds = rolling_origin_folds(origins, horizons, n_folds=3)
for i, (fit_o, val_o) in enumerate(folds):
    print(f"fold {i}: fit={len(fit_o)} origins ({fit_o[0]}-{fit_o[-1]}), val={val_o}")

fold_data = []
for fit_o, val_o in folds:
    fit_mat = build_direct_horizon_matrix(df, origins=fit_o, horizons=horizons)
    val_mat = build_direct_horizon_matrix(df, origins=val_o, horizons=horizons)
    fit_mat["segment"] = fit_mat["segment"].astype("category")
    val_mat["segment"] = val_mat["segment"].astype("category")
    fold_data.append((fit_mat, val_mat))

feature_cols = [
    c for c in fold_data[0][0].columns if c not in (*KEY, "origin_date", "target_date", "target")
]

fold 0: fit=5 origins (52-68), val=[84, 88, 92, 96, 100]
fold 1: fit=10 origins (52-88), val=[104, 108, 112, 116, 120]
fold 2: fit=15 origins (52-108), val=[124, 128, 132, 136, 140]


## 2. Re-confirm the objective choice -- exclusively VN1-driven, origin-mean early stopping

`train_lgbm(..., origin_col="origin_date")` uses `metric="None"` + `first_metric_only=True`
+ a mean-of-origin-scores eval function, and asserts only that metric was tracked. This
changed the picture substantially from the first two passes, which used `eval_metric="l1"`
then a pooled (not origin-mean) `vn1_lgb_eval` -- both of which turn out to have distorted
this comparison, not just the final headline number.

In [3]:
# Cell - objective comparison across all 3 folds, scored per-origin, exclusively VN1-driven
base_params = dict(n_estimators=2000, learning_rate=0.05, num_leaves=63, min_child_samples=50)
per_obj_origin_scores = {}
for objective in ["regression", "regression_l1", "tweedie"]:
    origin_scores = {}
    for fit_mat, val_mat in fold_data:
        _, preds = train_lgbm(fit_mat, val_mat, feature_cols, objective=objective,
                               categorical_feature=["segment"], origin_col="origin_date", **base_params)
        by_origin = vn1_score_by_group(val_mat["target"], preds, val_mat["origin_date"])
        origin_scores.update(by_origin.to_dict())
    per_obj_origin_scores[objective] = origin_scores
    print(f"{objective:15s} mean-of-origins={np.mean(list(origin_scores.values())):.4f}")

regression      mean-of-origins=0.5510


regression_l1   mean-of-origins=0.6503


tweedie         mean-of-origins=0.5490


**A real surprise: regression_l1 is still clearly worse, but tweedie is now nearly tied with
regression** (previously ~0.78 vs ~0.55 -- a huge gap that turns out to have been distorted by
the flawed early stopping in the first two passes). Comparing regression vs. tweedie origin by
origin, the same way 127-vs-200 leaves was checked, before trusting the aggregate mean.

In [4]:
# Cell - regression vs tweedie, origin by origin
reg, twd = per_obj_origin_scores["regression"], per_obj_origin_scores["tweedie"]
common = sorted(reg.keys())
diffs = [twd[o] - reg[o] for o in common]  # negative = tweedie better
for o in common:
    print(f"{o.date()}: regression={reg[o]:.4f}  tweedie={twd[o]:.4f}  diff(tweedie-regression)={twd[o]-reg[o]:+.4f}")
print(f"\nmean diff (tweedie-regression): {np.mean(diffs):+.4f}")
print(f"tweedie wins: {sum(d<0 for d in diffs)}/{len(diffs)}   regression wins: {sum(d>0 for d in diffs)}/{len(diffs)}")

2022-02-14: regression=0.4738  tweedie=0.4577  diff(tweedie-regression)=-0.0161
2022-03-14: regression=0.4823  tweedie=0.4775  diff(tweedie-regression)=-0.0048
2022-04-11: regression=0.5149  tweedie=0.5214  diff(tweedie-regression)=+0.0065
2022-05-09: regression=0.5150  tweedie=0.4891  diff(tweedie-regression)=-0.0259
2022-06-06: regression=0.4903  tweedie=0.5016  diff(tweedie-regression)=+0.0113
2022-07-04: regression=0.5038  tweedie=0.5230  diff(tweedie-regression)=+0.0193
2022-08-01: regression=0.5071  tweedie=0.4909  diff(tweedie-regression)=-0.0162
2022-08-29: regression=0.6049  tweedie=0.6144  diff(tweedie-regression)=+0.0095
2022-09-26: regression=0.6873  tweedie=0.7052  diff(tweedie-regression)=+0.0179
2022-10-24: regression=0.7055  tweedie=0.7352  diff(tweedie-regression)=+0.0298
2022-11-21: regression=0.6430  tweedie=0.5558  diff(tweedie-regression)=-0.0872
2022-12-19: regression=0.6033  tweedie=0.6138  diff(tweedie-regression)=+0.0105
2023-01-16: regression=0.5000  tweedie=0

**Decision: keep `regression`.** It wins the origin-count comparison (9/15 vs. tweedie's
6/15) and is the simpler choice (no extra variance-power hyperparameter) -- but the margin is
now known to be small (mean diff only -0.0020), not the large gap the first two passes
reported. That gap was a methodology artifact, not a real difference between the objectives.

## 3. Hyperparameter search, selected by mean-of-origin-level scores

In [5]:
# Cell - hyperparameter search
candidates = [
    dict(n_estimators=2000, learning_rate=0.05, num_leaves=63, min_child_samples=50),
    dict(n_estimators=2000, learning_rate=0.05, num_leaves=127, min_child_samples=50),
    dict(n_estimators=2000, learning_rate=0.05, num_leaves=200, min_child_samples=50),
    dict(n_estimators=2000, learning_rate=0.05, num_leaves=127, min_child_samples=30),
    dict(n_estimators=2000, learning_rate=0.03, num_leaves=127, min_child_samples=50),
]
per_config_origin_scores = {}
for params in candidates:
    origin_scores = {}
    for fit_mat, val_mat in fold_data:
        _, preds = train_lgbm(fit_mat, val_mat, feature_cols, categorical_feature=["segment"],
                               origin_col="origin_date", **params)
        by_origin = vn1_score_by_group(val_mat["target"], preds, val_mat["origin_date"])
        origin_scores.update(by_origin.to_dict())
    key = (params["num_leaves"], params["min_child_samples"], params["learning_rate"])
    per_config_origin_scores[key] = origin_scores
    print(f"{params}  mean-of-origins={np.mean(list(origin_scores.values())):.4f}")

{'n_estimators': 2000, 'learning_rate': 0.05, 'num_leaves': 63, 'min_child_samples': 50}  mean-of-origins=0.5510


{'n_estimators': 2000, 'learning_rate': 0.05, 'num_leaves': 127, 'min_child_samples': 50}  mean-of-origins=0.5482


{'n_estimators': 2000, 'learning_rate': 0.05, 'num_leaves': 200, 'min_child_samples': 50}  mean-of-origins=0.5459


{'n_estimators': 2000, 'learning_rate': 0.05, 'num_leaves': 127, 'min_child_samples': 30}  mean-of-origins=0.5498


{'n_estimators': 2000, 'learning_rate': 0.03, 'num_leaves': 127, 'min_child_samples': 50}  mean-of-origins=0.5442


In [6]:
# Cell - 127 vs 200 leaves, origin by origin (200 has the best mean, but it's also the
# largest value tested -- a search-boundary result worth distrusting on its own)
key_127, key_200 = (127, 50, 0.05), (200, 50, 0.05)
s127, s200 = per_config_origin_scores[key_127], per_config_origin_scores[key_200]
common_origins = sorted(s127.keys())
diffs = [s200[o] - s127[o] for o in common_origins]
wins_200 = sum(d < 0 for d in diffs)
wins_127 = sum(d > 0 for d in diffs)
print(f"mean diff (200-127): {np.mean(diffs):+.4f}   200 wins: {wins_200}/{len(diffs)}   127 wins: {wins_127}/{len(diffs)}")

mean diff (200-127): -0.0023   200 wins: 9/15   127 wins: 6/15


**Decision: 127, not 200.** Still practically tied (200 wins only 9/15, mean diff a tiny
amount) -- prefer the simpler, non-boundary configuration, per DECISIONS.md D009.

## 4. Sparse-segment investigation

Session 4 found the sparse segment contributes 21.6% of total absolute error from only 10.6%
of demand. Hypothesis: a model trained only on sparse-segment rows might specialize better.

**Note on scope**: this comparison is at the fold-pooled level (5 origins combined per fold),
not origin-by-origin like the comparisons above -- the resulting difference is small enough
(see below) that further disaggregation is unlikely to change the conclusion, but that has not
been separately verified the way 127-vs-200 and regression-vs-tweedie were.

In [7]:
# Cell - global model vs a dedicated sparse-only model, scored on the same sparse val rows
FINAL_PARAMS = dict(n_estimators=2000, learning_rate=0.05, num_leaves=127, min_child_samples=50)
global_sparse_scores, dedicated_sparse_scores = [], []
sparse_feats = [c for c in feature_cols if c != "segment"]

for fm, vm in fold_data:
    _, global_preds = train_lgbm(fm, vm, feature_cols, categorical_feature=["segment"],
                                  origin_col="origin_date", **FINAL_PARAMS)
    mask = (vm["segment"] == "sparse").to_numpy()
    global_sparse_scores.append(vn1_score(vm.loc[mask, "target"], global_preds[mask]))

    fit_sparse, val_sparse = fm[fm["segment"] == "sparse"], vm[vm["segment"] == "sparse"]
    _, sparse_preds = train_lgbm(fit_sparse, val_sparse, sparse_feats,
                                  origin_col="origin_date", **FINAL_PARAMS)
    dedicated_sparse_scores.append(vn1_score(val_sparse["target"], sparse_preds))

print(f"mean global-on-sparse: {np.mean(global_sparse_scores):.4f}")
print(f"mean dedicated-sparse: {np.mean(dedicated_sparse_scores):.4f}")

mean global-on-sparse: 1.2990
mean dedicated-sparse: 1.2929


**Revised conclusion.** Under the flawed early stopping in the first two passes, the dedicated
sparse model looked clearly worse (mean ~1.47 vs ~1.40). With exclusive-VN1, origin-mean early
stopping, the two are now nearly tied. **Decision unchanged in practice (keep one global
model)** -- the difference is too small to justify a second model's complexity -- but the
earlier claim of a clear negative result does not survive the corrected methodology and should
not be repeated as strong evidence.

## 5. LightGBM vs. MA4, origin by origin -- with bias diagnostics and a sensitivity check

MA4's forecast for a given origin is exactly that origin's own `roll_mean_4` feature.

In [8]:
# Cell - LightGBM vs MA4, scored separately for every one of the 15 rolling-CV origins
rows = []
for fit_mat, val_mat in fold_data:
    _, lgbm_preds = train_lgbm(fit_mat, val_mat, feature_cols, categorical_feature=["segment"],
                                origin_col="origin_date", **FINAL_PARAMS)
    ma4_preds = val_mat["roll_mean_4"].to_numpy()

    lgbm_by_origin = vn1_score_by_group(val_mat["target"], lgbm_preds, val_mat["origin_date"])
    ma4_by_origin = vn1_score_by_group(val_mat["target"], ma4_preds, val_mat["origin_date"])

    for origin in lgbm_by_origin.index:
        mask = (val_mat["origin_date"] == origin).to_numpy()
        actual_total = val_mat.loc[mask, "target"].sum()
        lgbm_bias = (val_mat.loc[mask, "target"] - lgbm_preds[mask]).sum()
        ma4_bias = (val_mat.loc[mask, "target"] - ma4_preds[mask]).sum()
        rows.append({
            "origin": origin, "lgbm_score": lgbm_by_origin[origin], "ma4_score": ma4_by_origin[origin],
            "diff": lgbm_by_origin[origin] - ma4_by_origin[origin],
            "lgbm_bias": lgbm_bias, "ma4_bias": ma4_bias, "actual_total": actual_total,
        })

report = pd.DataFrame(rows).sort_values("origin").reset_index(drop=True)
report

,origin,lgbm_score,ma4_score,diff,lgbm_bias,ma4_bias,actual_total
0,2022-02-14,0.507561,0.428319,0.079241,-63053.054533,75508.25,2406665.0
1,2022-03-14,0.474502,0.419040,0.055461,31022.340943,106145.25,2448872.0
2,2022-04-11,0.499843,0.464983,0.034860,36382.111376,110674.25,2511108.0
3,2022-05-09,0.531049,0.450043,0.081006,-118401.371113,79279.75,2546319.0
4,2022-06-06,0.507104,0.464229,0.042874,-53668.203637,118707.25,2580306.0
5,2022-07-04,0.501963,0.510860,-0.008897,60497.393893,112618.00,2588468.0
6,2022-08-01,0.501932,0.525598,-0.023666,-36942.197553,-58979.50,2604480.0
7,2022-08-29,0.600873,0.655598,-0.054725,98096.208015,278479.25,2944997.0
8,2022-09-26,0.687646,0.839959,-0.152313,135085.867882,972177.25,3380294.0
9,2022-10-24,0.698652,0.833163,-0.134511,210695.922246,894788.50,3548199.0


In [9]:
# Cell - win rate, mean/median diff, variability
n = len(report)
lgbm_wins = (report["diff"] < 0).sum()
ma4_wins = (report["diff"] > 0).sum()
print(f"n_origins={n}  LightGBM wins={lgbm_wins}  MA4 wins={ma4_wins}  ties={n-lgbm_wins-ma4_wins}")
print(f"mean diff (lgbm-ma4): {report['diff'].mean():+.4f}   median diff: {report['diff'].median():+.4f}")
print(f"std of diff: {report['diff'].std():.4f}")

n_origins=15  LightGBM wins=8  MA4 wins=7  ties=0
mean diff (lgbm-ma4): -0.0352   median diff: -0.0006
std of diff: 0.1515


In [10]:
# Cell - leave-one-out sensitivity: how much does a single origin drive the mean?
total_diff = report["diff"].sum()
report["loo_mean_excl_this_origin"] = [(total_diff - d) / (n - 1) for d in report["diff"]]

worst = report.loc[report["diff"].idxmax()]  # worst FOR LightGBM
best = report.loc[report["diff"].idxmin()]   # best FOR LightGBM
print(f"mean diff excluding worst-for-LightGBM origin ({worst['origin'].date()}): "
      f"{worst['loo_mean_excl_this_origin']:+.4f}")
print(f"mean diff excluding best-for-LightGBM origin ({best['origin'].date()}):  "
      f"{best['loo_mean_excl_this_origin']:+.4f}")

mean diff excluding worst-for-LightGBM origin (2022-05-09): -0.0435
mean diff excluding best-for-LightGBM origin (2022-12-19):  -0.0004


In [11]:
# Cell - bias diagnostics: absolute (not signed), median, normalized by demand, over/under counts
print(f"mean |bias|   -- LightGBM: {report['lgbm_bias'].abs().mean():,.0f}   MA4: {report['ma4_bias'].abs().mean():,.0f}")
print(f"median |bias| -- LightGBM: {report['lgbm_bias'].abs().median():,.0f}   MA4: {report['ma4_bias'].abs().median():,.0f}")

lgbm_norm = report["lgbm_bias"].abs() / report["actual_total"]
ma4_norm = report["ma4_bias"].abs() / report["actual_total"]
print(f"mean normalized |bias| (|bias|/demand)   -- LightGBM: {lgbm_norm.mean():.4f}   MA4: {ma4_norm.mean():.4f}")
print(f"median normalized |bias|                 -- LightGBM: {lgbm_norm.median():.4f}   MA4: {ma4_norm.median():.4f}")
print(f"over-forecasting origins (bias<0)  -- LightGBM: {(report['lgbm_bias']<0).sum()}   MA4: {(report['ma4_bias']<0).sum()}")
print(f"under-forecasting origins (bias>0) -- LightGBM: {(report['lgbm_bias']>0).sum()}   MA4: {(report['ma4_bias']>0).sum()}")

mean |bias|   -- LightGBM: 87,270   MA4: 270,009
median |bias| -- LightGBM: 70,980   MA4: 110,674
mean normalized |bias| (|bias|/demand)   -- LightGBM: 0.0286   MA4: 0.0850
median normalized |bias|                 -- LightGBM: 0.0234   MA4: 0.0435
over-forecasting origins (bias<0)  -- LightGBM: 5   MA4: 3
under-forecasting origins (bias>0) -- LightGBM: 10   MA4: 12


**Honest conclusion.** LightGBM wins slightly more origins now (8/15 vs. MA4's 7/15), and the
median diff is close to zero. But the leave-one-out check shows most of the mean advantage
still depends on one origin (2022-12-19, a year-end forecast origin where MA4 performed
exceptionally poorly -- seasonality is a plausible hypothesis for why, not established by this
comparison alone) -- exclude it and the mean advantage nearly vanishes. The one *robust,
consistent* LightGBM advantage, confirmed with absolute (not signed) and normalized bias
measures rather than a signed mean that can hide cancellation: bias control is genuinely
tighter, by roughly 3x on both mean and normalized absolute bias.

**Evidence is mixed. LightGBM provides substantial protection during one extreme period and
controls portfolio bias more tightly, but the overall origin-level margin is small and
concentrated rather than uniform. Whether that tail-risk advantage transfers to genuinely
unseen data is exactly what the real Phase 1 result will show.**

## 5b. Train-vs-validation diagnostic (overfitting vs. underfitting)

Everything above (early stopping, purged CV, the one holdout look) reduces overfitting risk
or checks whether the *selection process* generalizes -- none of it is the classic
train-score-vs-validation-score gap. Computed here directly, on the same 3 purged
rolling-origin folds and the same `FINAL_PARAMS` already chosen (D009) -- no retuning, no
new model.

Both scores use the mean-of-origin-level VN1 score (`vn1_score_by_group`, same convention as
the rest of this notebook) rather than a pooled score, on each fold's own fit origins (train)
and validation origins (val).

In [12]:
# Cell - train-vs-validation VN1 score per fold, using FINAL_PARAMS (no tuning here)
diag_rows = []
for fold_i, (fit_mat, val_mat) in enumerate(fold_data):
    model, val_preds = train_lgbm(fit_mat, val_mat, feature_cols, categorical_feature=["segment"],
                                   origin_col="origin_date", **FINAL_PARAMS)
    train_preds = np.clip(model.predict(fit_mat[feature_cols]), 0, None)

    train_by_origin = vn1_score_by_group(fit_mat["target"], train_preds, fit_mat["origin_date"])
    val_by_origin = vn1_score_by_group(val_mat["target"], val_preds, val_mat["origin_date"])

    train_score = train_by_origin.mean()
    val_score = val_by_origin.mean()
    diag_rows.append({
        "fold": fold_i,
        "n_train_origins": len(train_by_origin),
        "n_val_origins": len(val_by_origin),
        "train_score": train_score,
        "val_score": val_score,
        "gap_val_minus_train": val_score - train_score,
    })

diag = pd.DataFrame(diag_rows)
print(diag.to_string(index=False))
print(f"\nmean train_score: {diag['train_score'].mean():.4f}")
print(f"mean val_score:   {diag['val_score'].mean():.4f}")
print(f"mean gap (val-train): {diag['gap_val_minus_train'].mean():+.4f}")

 fold  n_train_origins  n_val_origins  train_score  val_score  gap_val_minus_train
    0                5              5     0.414389   0.504012             0.089622
    1               10              5     0.378760   0.598213             0.219453
    2               15              5     0.374958   0.542384             0.167425

mean train_score: 0.3894
mean val_score:   0.5482
mean gap (val-train): +0.1588


## 5c. Does regularizing close the train-vs-validation gap?

5b found a consistent overfitting signature (mean gap +0.1588; train 0.3894 beats every
baseline this project has ever scored on held-out data). Testing whether standard LightGBM
regularization -- fewer leaves, larger `min_child_samples`, L1/L2 penalties -- closes that gap
without hurting `val_score`. Same 3 purged folds, same mean-of-origin scoring as 5b.

This is observation, not a decision -- `FINAL_PARAMS` and DECISIONS.md are untouched here.

In [13]:
# Cell - regularization candidates vs. the 5b baseline, same train-vs-val diagnostic
reg_candidates = [
    ("baseline (FINAL_PARAMS)", dict(n_estimators=2000, learning_rate=0.05, num_leaves=127, min_child_samples=50)),
    ("larger min_child_samples", dict(n_estimators=2000, learning_rate=0.05, num_leaves=127, min_child_samples=300)),
    ("fewer leaves", dict(n_estimators=2000, learning_rate=0.05, num_leaves=31, min_child_samples=50)),
    ("L1+L2 penalty", dict(n_estimators=2000, learning_rate=0.05, num_leaves=127, min_child_samples=50,
                            reg_alpha=1.0, reg_lambda=10.0)),
    ("combined (leaves+samples+L1L2+subsample)", dict(n_estimators=2000, learning_rate=0.05, num_leaves=31,
                                                        min_child_samples=300, reg_alpha=1.0, reg_lambda=10.0,
                                                        colsample_bytree=0.8, subsample=0.8, subsample_freq=1)),
]

reg_rows = []
for label, params in reg_candidates:
    fold_train, fold_val = [], []
    for fit_mat, val_mat in fold_data:
        model, val_preds = train_lgbm(fit_mat, val_mat, feature_cols, categorical_feature=["segment"],
                                       origin_col="origin_date", **params)
        train_preds = np.clip(model.predict(fit_mat[feature_cols]), 0, None)
        fold_train.append(vn1_score_by_group(fit_mat["target"], train_preds, fit_mat["origin_date"]).mean())
        fold_val.append(vn1_score_by_group(val_mat["target"], val_preds, val_mat["origin_date"]).mean())
    train_score, val_score = np.mean(fold_train), np.mean(fold_val)
    reg_rows.append({
        "config": label, "train_score": train_score, "val_score": val_score,
        "gap_val_minus_train": val_score - train_score,
    })

reg_diag = pd.DataFrame(reg_rows)
pd.set_option("display.max_colwidth", 60)
print(reg_diag.to_string(index=False))

                                  config  train_score  val_score  gap_val_minus_train
                 baseline (FINAL_PARAMS)     0.389369   0.548203             0.158834
                larger min_child_samples     0.450463   0.548462             0.097999
                            fewer leaves     0.465889   0.554413             0.088523
                           L1+L2 penalty     0.383597   0.544649             0.161052
combined (leaves+samples+L1L2+subsample)     0.481004   0.552350             0.071346


## 5d. Training-size learning curve with a fixed validation block

Keep the final rolling-validation block fixed as the exam and use the baseline
`FINAL_PARAMS` throughout. The fit sets are nested: each larger set retains the same
latest eligible fit origin and adds older, properly purged origins. This isolates the
effect of adding historical training examples without moving the validation period,
changing features, tuning parameters, or inspecting the final 13-week holdout.


In [14]:
# Cell - fixed-exam training-size learning curve, baseline FINAL_PARAMS only
fixed_val_origins = folds[-1][1]
eligible_fit_origins = purge_fit_origins(origins, fixed_val_origins, horizons)
assert max(eligible_fit_origins) + max(horizons) < min(fixed_val_origins)
fixed_val_mat = fold_data[-1][1]

training_sizes = [3, 5, 8, 10, 12, len(eligible_fit_origins)]
curve_rows = []
for n_fit_origins in training_sizes:
    # Keep the training endpoint fixed; add progressively older history.
    fit_o = eligible_fit_origins[-n_fit_origins:]
    fit_mat = build_direct_horizon_matrix(df, origins=fit_o, horizons=horizons)
    fit_mat["segment"] = fit_mat["segment"].astype("category")
    model, val_preds = train_lgbm(
        fit_mat, fixed_val_mat, feature_cols, categorical_feature=["segment"],
        origin_col="origin_date", **FINAL_PARAMS
    )
    train_preds = np.clip(model.predict(fit_mat[feature_cols]), 0, None)
    train_score = vn1_score_by_group(
        fit_mat["target"], train_preds, fit_mat["origin_date"]
    ).mean()
    val_score = vn1_score_by_group(
        fixed_val_mat["target"], val_preds, fixed_val_mat["origin_date"]
    ).mean()
    curve_rows.append({
        "fit_origins": n_fit_origins, "training_rows": len(fit_mat),
        "train_score": train_score, "val_score": val_score,
        "gap_val_minus_train": val_score - train_score,
    })

learning_curve = pd.DataFrame(curve_rows)
print("fixed validation origins:", fixed_val_origins)
print(learning_curve.to_string(index=False, formatters={
    "train_score": "{:.4f}".format, "val_score": "{:.4f}".format,
    "gap_val_minus_train": "{:+.4f}".format,
}))


fixed validation origins: [124, 128, 132, 136, 140]
 fit_origins  training_rows train_score val_score gap_val_minus_train
           3         587067      0.4204    0.6465             +0.2261
           5         978445      0.4258    0.6070             +0.1812
           8        1565512      0.3629    0.5567             +0.1938
          10        1956890      0.3182    0.5368             +0.2186
          12        2348268      0.3470    0.5437             +0.1967
          15        2935335      0.3750    0.5424             +0.1674


## 6. Final evaluation on the development holdout

Session 5 performs one final evaluation on the already-used Phase 0 development holdout; it
is not a pristine test set -- it has been inspected in Sessions 3, 4, and two earlier passes
of this session.

In [15]:
# Cell - retrain the final configuration, evaluate once more, with the full metric-exclusivity check
val_origins = origins[-3:]
fit_origins = purge_fit_origins(origins, val_origins, horizons)
train_mat = build_direct_horizon_matrix(df, origins=fit_origins, horizons=horizons)
val_mat = build_direct_horizon_matrix(df, origins=val_origins, horizons=horizons)
eval_mat = build_direct_horizon_matrix(df, origins=[156], horizons=horizons)
for m in (train_mat, val_mat, eval_mat):
    m["segment"] = m["segment"].astype("category")

model, _ = train_lgbm(train_mat, val_mat, feature_cols, categorical_feature=["segment"],
                       origin_col="origin_date", **FINAL_PARAMS)
print("best_iteration_:", model.best_iteration_)
print("tracked metric (must be exclusively VN1-based):", list(model.evals_result_["valid_0"]))
print("full params:", model.get_params())

best_iteration_: 491
tracked metric (must be exclusively VN1-based): ['vn1_score_origin_mean']
full params: {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 50, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 2000, 'n_jobs': None, 'num_leaves': 127, 'objective': 'regression', 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'metric': 'None', 'verbosity': -1}


In [16]:
# Cell - predict, clip, align, score
preds = np.clip(model.predict(eval_mat[feature_cols]), 0, None)
forecast = eval_mat[list(KEY) + ["target_date"]].copy()
forecast["forecast"] = preds
forecast = forecast.rename(columns={"target_date": "date"})

scored = score_forecast(test, forecast)
breakdown = vn1_score_breakdown(scored["sales"], scored["forecast"])
for k, v in breakdown.items():
    print(f"{k:15s}: {v:,.4f}" if k != "score" else f"{k:15s}: {v:.4f}")

total_actual   : 3,508,827.0000
total_predicted: 3,467,480.9584
signed_bias    : 41,346.0416
abs_error_sum  : 1,711,493.8401
bias_term      : 41,346.0416
score          : 0.4996


## 7. Compare against the full baseline suite

In [17]:
# Cell - final comparison
results = {"LightGBM (Session 5, fully corrected)": breakdown["score"]}
scored_zero = score_forecast(test, zero_forecast(train))
results["zero"] = vn1_score(scored_zero["sales"], scored_zero["forecast"])
for name, window in [("naive-last", 1), ("MA4", 4), ("MA8", 8), ("MA12", 12), ("MA26", 26)]:
    sb = score_forecast(test, ma12_forecast(train, window=window))
    results[name] = vn1_score(sb["sales"], sb["forecast"])
sn = score_forecast(test, seasonal_naive_forecast(df, test, lag_weeks=52))
results["seasonal-naive-52"] = vn1_score(sn["sales"], sn["forecast"])

pd.Series(results, name="vn1_score").sort_values()

MA4                                      0.498708
LightGBM (Session 5, fully corrected)    0.499552
MA8                                      0.511414
MA12                                     0.516309
naive-last                               0.524666
MA26                                     0.561836
seasonal-naive-52                        1.049790
zero                                     2.000000
Name: vn1_score, dtype: float64

**On this single development-holdout point, MA4 very slightly edges out LightGBM** -- the
opposite of what the rolling-CV win-rate (8/15 favoring LightGBM) would suggest. This is
consistent with the honest conclusion above: the evidence is mixed and the margin is small
either way, not a case of one model dominating. This single point should not be read as
stronger evidence than the 15-origin rolling comparison -- if anything, it's the noisier of
the two estimates, and is a development holdout, not a pristine one.